# Unit 10 exercises: approximate nearest neighbours (ANN) and hybrid retrieval

Run the recsys data generators before you begin. Every dense path the course has built -- Unit 7 semantic, Unit 8 two-tower, Unit 9 feature tower -- retrieves the same way: score **all** 2,000 books for a reader by the `reader . item` dot product and sort. That is exact, and it is **linear in the catalog**: fine at 2,000 books, hopeless at a million. This unit is about the **retrieval infrastructure**, not a new model. You will build an **approximate** index -- a FAISS **HNSW** (Hierarchical Navigable Small World) graph -- over the Unit-8 two-tower item vectors, measure how closely it recovers the exact top-10 (**recall@10 vs exact**), and watch that recall rise as you spend more search effort (`efSearch`).

Two honest headlines frame the unit. First, **ANN preserves accuracy**: it recovers (approximately) the same top-10 the exact scan would, so its hit@10 is identical to the exact two-tower (~0.340). ANN is a **speed** technique, not an accuracy one -- and at 2,000 items there is **no speedup at all** (brute force over 2000x32 is already microseconds); the win is **asymptotic** (a 20k index is ~10x faster). Second, you will build a **hybrid** path that fuses the sparse lexical scores (Unit 3 BM25) with the dense two-tower scores, and read it honestly: the tuned dense-heavy fusion (`pool=50, weight=0.7`) edges the two-tower on both hit@10 (~0.362) and coverage (~0.192 vs 0.177), but the lift is **small and tuning-dependent** -- equal weights *hurt* and reciprocal-rank fusion *loses*.

Everything is measured on the **validation** split (`val`); each reader's **seen** set is their train-positive history. The sealed `test` holdout stays sealed -- no cell here may touch it. The index is built single-threaded (`faiss.omp_set_num_threads(1)`, saved and restored inside `HnswIndex`) and ranks by **inner product** (`METRIC_INNER_PRODUCT`), so it ranks the *same* dot product the two-tower's exact scan uses. Each answer is left for you to complete; no solutions live here.

A speed note, because fitting recurs. The Unit-8 two-tower fit this unit reuses is ~13-15 s; it is fitted **once** in the setup cell and every exercise reuses it. The HNSW build at 2,000 items is ~0.16 s and at 20k ~2.5 s, so every cell stays well under the 120 s/cell budget.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    AnnRetrievalPath,
    HnswIndex,
    HybridRetrievalPath,
    LexicalRetrievalPath,
    TwoTowerRetrievalPath,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

SEED = 0
K = 10                  # every scoreboard reports at the top-10 cutoff
M = 32                  # HNSW graph degree (pinned)
EF_CONSTRUCTION = 200   # build-time graph exploration (pinned)
EF_SEARCH = 64          # query-time exploration knob (pinned: recall@10 vs exact ~0.999)

# --- the real interaction log + content (the same substrate as Units 1-9) ------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")        # {item_id: Book} (2000 books)
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")     # {item_id: "space joined tokens"}

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])      # readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

# Each reader's train-positive history is their `seen` (exclusion) set. We grade on `val`; no cell
# here may read the sealed `test` split.
seen_by = defaultdict(set)
for row in rows:
    if row["label"] == 1 and row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])

# The Unit-8 two-tower is the dense SOURCE every path in this unit reuses -- one ~13-15 s fit.
# Its reader and item vectors live in ONE inner-product space (hit@10 ~0.340, the warm leader).
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)
tt_art = two_tower.artifact()
item_matrix = tt_art["item_embeddings"]       # (2000, 32) float32 -- the vectors the index will hold
reader_matrix = tt_art["reader_embeddings"]   # (n_readers, 32) float32 -- the query vectors
item_ids = tt_art["item_ids"]                 # item id aligned to each item_matrix ROW
reader_ids = tt_art["reader_ids"]             # reader id aligned to each reader_matrix ROW

## Exercise 1 - Build an HNSW index over the two-tower vectors and query it

The exact brute-force baseline is what every dense path already does: score every item by the `reader . item` dot product and sort -- `O(catalog)` work per reader. An **approximate** index trades a little recall for sublinear query time at scale by visiting only a *few* promising neighbours in a navigable graph instead of touching every item.

`HnswIndex(items, ids, m=M, ef_construction=EF_CONSTRUCTION)` builds that graph over the two-tower item matrix. The one detail that must be right is the **metric**: the index is built `IndexHNSWFlat(dim, M, METRIC_INNER_PRODUCT)` so it ranks the **same inner product** the two-tower's exact scan uses -- FAISS's *default* is L2 distance, which would rank differently and make recall-vs-exact meaningless. `index.search(queries, k, efSearch=EF_SEARCH)` returns `(labels, scores)`, where `labels` are **row positions** into the item matrix (a `-1` pads a row that found fewer than `k` neighbours).

Write the exact brute-force top-k for one reader, build the index, query it for that same reader, and confirm the approximate neighbours recover almost all of the exact top-10. In two or three sentences, explain why the index must use `METRIC_INNER_PRODUCT` rather than the FAISS default.

In [ ]:
# The EXACT brute-force baseline: score every item by the reader.item dot product and sort.
# This is what every dense path already does -- linear in the catalog.
def exact_top_k(reader_vec, k):
    scores = reader_vec @ item_matrix.T
    order = np.argsort(-scores)                 # descending by score
    return [item_ids[row] for row in order[:k]]


# Build the approximate index over the SAME item vectors, ranking by inner product.
index = HnswIndex(item_matrix, item_ids, m=M, ef_construction=EF_CONSTRUCTION)

one_row = 0                                     # the first fitted reader (a row into reader_matrix)
reader_vec = reader_matrix[one_row]

exact_ids = None    # TODO: exact_top_k(reader_vec, K)   -- the brute-force top-10 item ids
# TODO: index.search needs a 2-D batch of queries; give it one row via [np.newaxis, :].
labels, scores = None, None   # TODO: index.search(reader_matrix[one_row][np.newaxis, :], K, efSearch=EF_SEARCH)
ann_ids = None      # TODO: [item_ids[row] for row in labels[0] if row >= 0]   -- ANN top-10 item ids

index_size = None   # TODO: index.size   -- 2000 indexed items
overlap = None      # TODO: len(set(exact_ids) & set(ann_ids))   -- exact top-10 the ANN recovered
mostly_agree = None # TODO: bool - overlap >= 9   (ANN recovers almost all of the exact top-10 at ef=64)

metric_explanation = "TODO"  # 2-3 sentences: the index must rank by METRIC_INNER_PRODUCT so it scores
                             # the SAME reader.item dot product the exact scan uses; the FAISS default
                             # (L2 distance) would rank by a different geometry and recall-vs-exact
                             # would be meaningless

## Exercise 2 - Recall@10 vs exact, and watch it rise with `efSearch`

How faithfully does the approximate index recover the exact top-10? The measure is **recall@10 vs exact**: for each reader, the fraction of their exact top-10 that the ANN search also returns, averaged over readers. (This is recall *against the brute-force baseline*, not against held-out relevance -- it asks "did the approximation find what the exact scan would have found?")

The knob is `efSearch`: it controls how much of the graph a query explores. A higher `efSearch` visits more candidate neighbours -- higher recall, slower query. Sweep it across `8, 16, 32, 64, 128` and confirm recall rises from the cheapest to the most thorough search, and that the pinned `efSearch=64` already recovers essentially the whole exact top-10 (~0.999). Gate the trend on the **endpoints** (recall can plateau or tie near 1.0, so do not demand strict per-step monotonicity). In two or three sentences, explain what `efSearch` trades.

In [ ]:
# Exact top-10 ROW indices for EVERY reader (the brute-force reference), in item_matrix row space.
def recall_vs_exact(idx, matrix, ef):
    """Mean per-reader overlap@10 of idx's ANN top-10 with the exact top-10 (same inner-product scan)."""
    exact_rows = np.argsort(-(reader_matrix @ matrix.T), axis=1)[:, :K]
    labels, _ = idx.search(reader_matrix, K, efSearch=ef)
    overlaps = []
    for i in range(reader_matrix.shape[0]):
        ann_rows = {int(r) for r in labels[i] if r >= 0}
        exact_set = {int(r) for r in exact_rows[i]}
        overlaps.append(len(ann_rows & exact_set) / K)
    return float(np.mean(overlaps))


ef_sweep = [8, 16, 32, 64, 128]
recall_by_ef = None    # TODO: {ef: recall_vs_exact(index, item_matrix, ef) for ef in ef_sweep}
                       #       -- ~0.981 (ef=8) rising to ~1.000 (ef=128)
recall_rises = None    # TODO: bool - recall_by_ef[8] <= recall_by_ef[128]   (endpoints, not per-step)
recall_at_pinned = None  # TODO: recall_by_ef[EF_SEARCH]   -- ~0.999 at the pinned efSearch=64
pinned_is_high = None    # TODO: bool - recall_at_pinned >= 0.98

efsearch_explanation = "TODO"  # 2-3 sentences: efSearch controls how much of the navigable graph a
                               # query explores; a higher value visits more candidate neighbours, so
                               # recall rises toward the exact top-10 -- at a higher per-query cost

## Exercise 3 - ANN preserves accuracy: hit@10 == the exact two-tower

Recall-vs-exact says the ANN neighbours *are* the exact neighbours almost every time. The consequence is the headline of the unit: wrapping the two-tower in an ANN path does **not change its recommendations**, so its **hit@10 on `val` is identical to the exact two-tower** (~0.340, within a tiny tolerance for the handful of readers whose top-10 the approximation reorders).

`AnnRetrievalPath(dense_path, efSearch=..., m=..., ef_construction=...)` wraps an already-fit dense path: its `fit` builds the HNSW index over the dense source's item matrix (the `interactions` argument is ignored -- the two-tower is already trained), and its `retrieve` answers by an approximate index search instead of the exact full scan. Build it over the two-tower, score both paths on `val`, and confirm the hit@10 matches. In two or three sentences, explain why ANN is a **speed** technique and not an accuracy one.

In [ ]:
# Wrap the two-tower: AnnRetrievalPath.retrieve approximates the exact scan via the HNSW index.
# fit ignores `interactions` (the two-tower is already trained); it just builds the index.
ann = AnnRetrievalPath(two_tower, efSearch=EF_SEARCH, m=M, ef_construction=EF_CONSTRUCTION).fit(rows)


def val_hit(path):
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    ).hit_rate_at_k


tt_hit = None     # TODO: val_hit(two_tower)   -- ~0.340 (the exact brute-force two-tower)
ann_hit = None    # TODO: val_hit(ann)         -- ~0.340 (identical -- ANN preserves accuracy)
ann_matches_exact = None  # TODO: bool - abs(ann_hit - tt_hit) <= 0.005

accuracy_explanation = "TODO"  # 2-3 sentences: ANN recovers (approximately) the SAME top-10 the exact
                               # scan would, so its hit@10 equals the two-tower's; ANN buys SPEED at
                               # scale, not better recommendations -- a higher hit@10 from "adding ANN"
                               # would be a misreading

## Exercise 4 - The retrieve contract: over-fetch past `seen`, and unknown readers

An approximate index has the same contract an exact path does, and one subtlety that is easy to get wrong. A reader has a `seen` set (their train history) that must be **excluded** from recommendations -- a re-read is not a recommendation. If the index returned exactly `k` neighbours and then you dropped the seen ones, a reader who has already read several of their top neighbours would end up with *fewer* than `k` recs -- breaking parity with the exact scan. So `AnnRetrievalPath.retrieve` **over-fetches** `min(n_items, k + len(seen))` neighbours from the index *before* excluding `seen`. A reader with no dense vector (unknown / cold) returns `[]`.

Confirm the contract on the `ann` path from Exercise 3: a known reader with empty `seen` gets `K` recs; the same reader with 25 excluded items *still* gets `K` recs and none of the excluded ids appear; an unknown reader gets `[]`. In two or three sentences, explain why over-fetching past `seen` is what keeps ANN at parity with the exact scan for heavy readers.

In [ ]:
contract_reader = reader_ids[0]

empty_seen_recs = None   # TODO: ann.retrieve(contract_reader, {"seen": set()}, K)
got_k = None             # TODO: bool - len(empty_seen_recs) == K

# Exclude 25 catalog items as this reader\'s "seen": none may appear, and we still get K recs.
heavy_seen = set(catalog_ids[:25])
heavy_recs = None        # TODO: ann.retrieve(contract_reader, {"seen": heavy_seen}, K)
heavy_got_k = None       # TODO: bool - len(heavy_recs) == K
no_seen_leak = None      # TODO: bool - all(c.item_id not in heavy_seen for c in heavy_recs)

# An unknown reader (no row in reader_matrix) returns [].
unknown_reader = None    # TODO: next(r for r in range(10_000_000) if r not in set(reader_ids))
unknown_empty = None     # TODO: bool - ann.retrieve(unknown_reader, {"seen": set()}, K) == []

contract_explanation = "TODO"  # 2-3 sentences: over-fetching k + len(seen) BEFORE excluding seen means
                               # a reader who has read many of their top neighbours still gets k UNSEEN
                               # recs -- parity with the exact scan; an unknown reader has no reader
                               # vector, so there is nothing to query and the path returns []

## Exercise 5 - Determinism: two single-thread builds give the same index

A retrieval index you cannot reproduce is not science. HNSW construction is **thread-order dependent**: build it multi-threaded and the insertion order -- and so the graph -- varies run to run. `HnswIndex` builds single-threaded (`faiss.omp_set_num_threads(1)`, saved and restored in a `finally` block, exactly as `two_tower.fit` guards `torch.set_num_threads`), so two builds over the same matrix give the **same** approximate neighbours.

The gate here (design 011's determinism law) is **identical neighbour ids + `allclose` scores** on two independent single-thread builds -- *not* exact-float equality (FAISS returns `float32` inner products), and *not* exact neighbour identity against the brute force (ANN is approximate by construction, which is exactly what Exercises 1-2 measured). Build two indices, query both, and confirm they agree. In two or three sentences, explain why we gate on identical ids + `allclose` rather than on matching the exact brute-force neighbours.

In [ ]:
# Two independently-built single-thread indices over the SAME item matrix.
first = HnswIndex(item_matrix, item_ids, m=M, ef_construction=EF_CONSTRUCTION)
second = HnswIndex(item_matrix, item_ids, m=M, ef_construction=EF_CONSTRUCTION)

labels_a, scores_a = None, None   # TODO: first.search(reader_matrix, K, efSearch=EF_SEARCH)
labels_b, scores_b = None, None   # TODO: second.search(reader_matrix, K, efSearch=EF_SEARCH)

ids_identical = None     # TODO: bool - np.array_equal(labels_a, labels_b)   (the gate: same neighbours)
scores_allclose = None   # TODO: bool - np.allclose(scores_a, scores_b)      (the gate: float32 scores)

determinism_explanation = "TODO"  # 2-3 sentences: a single-thread build fixes the insertion order and
                                  # so the graph, making two builds identical; we gate on identical
                                  # neighbour ids + allclose scores (not exact floats), and NOT on
                                  # matching the brute force -- ANN is approximate by design

## Exercise 6 - A hybrid path: fuse BM25 + the two-tower, read hit@10 AND coverage

Unit 3's BM25 lexical path (sparse, keyword overlap) and the two-tower (dense, learned taste) surface largely *different* books -- lexical catches an on-the-nose keyword match the dense tower ranks low; the dense tower catches a taste match no keyword shares. `HybridRetrievalPath(sparse_path, dense_path, weight=0.7, pool=50, method="weighted")` **fuses** them: each path over-fetches its top `pool` candidates, each path's scores are min-max normalised over its own pool (so the two very different score scales become comparable), and the fused score is `weight * dense + (1 - weight) * sparse`. The pinned `weight=0.7` is **dense-heavy** because the dense tower is the stronger path.

Score BM25-alone, the two-tower-alone, and the hybrid on `val`, reading **both** hit@10 and catalog coverage. The honest story: the dense tower **dominates** (0.340 vs BM25 0.158), and the tuned hybrid gives a **small** lift on both hit@10 (~0.362) and coverage (~0.192 vs 0.177) -- real, but small and tuning-dependent, **not** a free win. In three or four sentences, state the ordering, say the hybrid's lift is small and dense-heavy, and explain what BM25 contributes.

In [ ]:
# Unit 3's BM25 lexical path (sparse): a reader\'s query is the words of the books they have read.
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)

# The pinned dense-heavy hybrid: fuse BM25 + the two-tower over each path\'s top-50 pool (w_dense=0.7).
hybrid = HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50, method="weighted").fit(rows)


def board(path):
    """hit@10 + catalog coverage via the single-path blended scoreboard (reproduces the path\'s hit@k)."""
    result = run_blended_scoreboard(
        {path.name: path}, interactions_path,
        catalog_ids=catalog_ids, k=K, pool=K, cold_readers=cold_readers,
    )
    return result.hit_rate_at_k, result.catalog_coverage


bm25_hit, bm25_cov = None, None      # TODO: board(lexical)     -- ~0.158 / ~0.278
tt_hit2, tt_cov = None, None         # TODO: board(two_tower)   -- ~0.340 / ~0.177
hybrid_hit, hybrid_cov = None, None  # TODO: board(hybrid)      -- ~0.362 / ~0.192

dense_dominates = None       # TODO: bool - tt_hit2 > bm25_hit   (the dense tower is the stronger path)
hybrid_edges_hit = None      # TODO: bool - hybrid_hit >= tt_hit2 - 0.01   (directional, measured-safe)
hybrid_lifts_coverage = None # TODO: bool - hybrid_cov > tt_cov

hybrid_reading = "TODO"  # 3-4 sentences: the dense tower dominates (0.340 vs BM25 0.158); the tuned
                         # dense-heavy hybrid edges it on BOTH hit (0.362) and coverage (0.192 > 0.177),
                         # but the lift is SMALL and tuning-dependent, not a free win; BM25 surfaces
                         # lexically-matched books the dense tower ranks low, which lifts coverage

## Exercise 7 - Challenge: recall across the catalog, and why speed is asymptotic

You measured recall rising with `efSearch` at 2,000 items. Two things change as the **catalog** grows. First, at a *fixed* `efSearch` recall **falls**: a bigger graph needs more exploration to find the true top-10, so the knob must scale with the catalog. Second -- the point of ANN -- the **speed** win is asymptotic: at 2,000 items brute force over `2000 x 32` is already microseconds and HNSW is no faster, but at tens of thousands of items the graph visits only a handful of nodes while brute force must touch every item (~10x faster at 20k).

Build a 20k synthetic index by stacking ten lightly-jittered copies of the item matrix (seeded, deterministic), measure recall@10 vs exact at the pinned `efSearch=64`, and confirm it is **lower** than the 2k recall at the same `efSearch` (reuse `recall_by_ef` from Exercise 2). In three or four sentences, describe both effects -- recall falling at fixed `efSearch`, and why the speed win only shows at scale.

In [ ]:
# A 20k synthetic catalog: ten lightly-jittered copies of the 2000 item vectors (seeded).
# HnswIndex coerces to contiguous float32 internally, so we can pass the stacked array directly.
rng = np.random.default_rng(SEED)
copies = [item_matrix + 0.01 * rng.normal(size=item_matrix.shape) for _ in range(10)]
big_matrix = np.concatenate(copies)             # (20000, 32)
big_ids = list(range(big_matrix.shape[0]))
big_index = HnswIndex(big_matrix, big_ids, m=M, ef_construction=EF_CONSTRUCTION)

big_recall = None    # TODO: recall_vs_exact(big_index, big_matrix, EF_SEARCH)   -- ~0.993 at 20k
small_recall = None  # TODO: recall_by_ef[EF_SEARCH]   -- ~0.999 at 2k (reuse Exercise 2)
recall_falls_with_size = None  # TODO: bool - big_recall < small_recall   (same efSearch, bigger catalog)
big_index_size = None  # TODO: big_index.size   -- 20000 indexed items

scaling_explanation = "TODO"  # 3-4 sentences: at a FIXED efSearch recall falls as the catalog grows
                              # (~0.999 @2k -> ~0.993 @20k), so the knob must scale with the catalog;
                              # the SPEED win is asymptotic -- no speedup at 2k (brute force is already
                              # microseconds), ~10x at 20k -- which is exactly when ANN earns its place

## Exercise 8 - Challenge: fusion is not free -- equal weights hurt and RRF loses

The hybrid in Exercise 6 used one tuned setting. Is fusion *generally* a win? No -- and this is the honest lesson. Two natural alternatives both **lose** to the two-tower alone on this data: **equal weights** (`weight=0.5`, giving the weaker BM25 path as much say as the dense tower) and **reciprocal-rank fusion** (`method="rrf"`, which fuses by *rank* instead of normalised score). Only the tuned, dense-heavy weighted sum gives the small lift you measured.

Score the pinned hybrid (`weight=0.7`), an equal-weight hybrid (`weight=0.5`), and an RRF hybrid, and compare each to the two-tower-alone hit@10 (`tt_hit2` from Exercise 6). Confirm that equal weights and RRF both come in **below** the two-tower, while only the tuned weight holds its lift. In three or four sentences, explain why naive fusion fails here -- what min-max normalisation plus a tuned, dense-heavy weight does that equal weights and RRF do not.

In [ ]:
def hybrid_hit(weight, pool, method):
    """hit@10 of a hybrid at the given fusion settings (reuse board() from Exercise 6)."""
    path = HybridRetrievalPath(lexical, two_tower, weight=weight, pool=pool, method=method).fit(rows)
    hit, _coverage = board(path)
    return hit


pinned_hit = None   # TODO: hybrid_hit(0.7, 50, "weighted")   -- ~0.362 (tuned dense-heavy; the lift)
equal_hit = None    # TODO: hybrid_hit(0.5, 50, "weighted")   -- ~0.30  (equal weights HURT)
rrf_hit = None      # TODO: hybrid_hit(0.5, 50, "rrf")        -- ~0.31-0.33 (RRF loses)

# Reuse tt_hit2 (the two-tower-alone hit@10) from Exercise 6.
equal_weights_hurt = None  # TODO: bool - equal_hit < tt_hit2
rrf_loses = None           # TODO: bool - rrf_hit < tt_hit2
only_tuned_helps = None    # TODO: bool - pinned_hit >= tt_hit2 - 0.01 and equal_weights_hurt and rrf_loses

fusion_explanation = "TODO"  # 3-4 sentences: equal weights and standard RRF both come in BELOW the
                             # two-tower alone; only the tuned dense-heavy weight (0.7) holds the small
                             # lift; fusing two score distributions on very different scales needs
                             # min-max normalisation PLUS a tuned weight -- naive fusion is not safe